In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.stats import skew, kurtosis
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem, DataStructs
from itertools import combinations
from itertools import combinations
from collections import Counter
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import pubchempy as pcp
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
plt.rcParams['font.sans-serif'] = ["Arial"]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.feature_selection import RFE
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
from tqdm import tqdm

# 读取数据
df = pd.read_csv("../../data/source_data/property_models/DM.csv")
df_x = pd.read_csv("df_final.csv")
X = df_x.iloc[:, 1:].values
feature_names = df_x.columns[1:]
y = df["DM"].values

# 参数设置
k = 10
kf = KFold(n_splits=k, shuffle=True, random_state=42)
rr_params = {
    "alpha": [0.01, 0.1, 1, 2, 5,],
    "tol": [0.01, 0.005, 1e-2, 1e-3]
}

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 存储结果
feature_counts = []
spearman_means, spearman_stds = [], []
rmse_means, rmse_stds = [], []
mae_means, mae_stds = [], []
r2_means, r2_stds = [], []
results = []
all_param_results = []

# 全局最优记录
best_spearman = -np.inf
best_selector = None
best_params = None
best_selected_features = None
best_train_idx, best_test_idx = None, None

# 进度条
pbar = tqdm(total=len(range(5, X.shape[1] + 1, 5)) * len(rr_params["alpha"]) * len(rr_params["tol"]))

# 主循环
for num_features in range(5, X.shape[1] + 1, 5):
    best_spearman_curr = -np.inf
    best_std_spearman = None
    best_rmse_curr, best_std_rmse = None, None
    best_mae_curr, best_std_mae = None, None
    best_r2_curr, best_std_r2 = None, None
    best_features_curr = None

    for alpha in rr_params["alpha"]:
        for tol in rr_params["tol"]:
            model = Ridge(alpha=alpha, tol=tol)
            selector = RFE(model, n_features_to_select=num_features, step=5)
            X_selected = selector.fit_transform(X_scaled, y)

            spearman_scores, rmse_scores, mae_scores, r2_scores = [], [], [], []
            fold_spearman_list = []

            for train_idx, val_idx in kf.split(X_selected):
                X_train, X_val = X_selected[train_idx], X_selected[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                model.fit(X_train, y_train)
                y_pred = model.predict(X_val)

                spearman_corr, _ = spearmanr(y_val, y_pred)
                rmse = mean_squared_error(y_val, y_pred, squared=False)
                mae = mean_absolute_error(y_val, y_pred)
                r2 = r2_score(y_val, y_pred)

                spearman_scores.append(spearman_corr)
                rmse_scores.append(rmse)
                mae_scores.append(mae)
                r2_scores.append(r2)
                fold_spearman_list.append((spearman_corr, train_idx, val_idx))

            spearman_mean, spearman_std = np.mean(spearman_scores), np.std(spearman_scores)
            rmse_mean, rmse_std = np.mean(rmse_scores), np.std(rmse_scores)
            mae_mean, mae_std = np.mean(mae_scores), np.std(mae_scores)
            r2_mean, r2_std = np.mean(r2_scores), np.std(r2_scores)

            all_param_results.append({
                "Num_Features": num_features,
                "Alpha": alpha,
                "Tol": tol,
                "Spearman_Mean": spearman_mean,
                "Spearman_Std": spearman_std,
                "RMSE_Mean": rmse_mean,
                "RMSE_Std": rmse_std,
                "MAE_Mean": mae_mean,
                "MAE_Std": mae_std,
                "R2_Mean": r2_mean,
                "R2_Std": r2_std,
                "Selected_Features": ",".join(feature_names[selector.support_])
            })

            if spearman_mean > best_spearman_curr:
                best_spearman_curr = spearman_mean
                best_std_spearman = spearman_std
                best_rmse_curr = rmse_mean
                best_std_rmse = rmse_std
                best_mae_curr = mae_mean
                best_std_mae = mae_std
                best_r2_curr = r2_mean
                best_std_r2 = r2_std
                best_features_curr = feature_names[selector.support_]

                if spearman_mean > best_spearman:
                    best_spearman = spearman_mean
                    best_selector = selector
                    best_params = {"alpha": alpha, "tol": tol}
                    best_selected_features = best_features_curr
                    best_fold = max(fold_spearman_list, key=lambda x: x[0])
                    best_train_idx, best_test_idx = best_fold[1], best_fold[2]

            pbar.update(1)

    feature_counts.append(num_features)
    spearman_means.append(best_spearman_curr)
    spearman_stds.append(best_std_spearman)
    rmse_means.append(best_rmse_curr)
    rmse_stds.append(best_std_rmse)
    mae_means.append(best_mae_curr)
    mae_stds.append(best_std_mae)
    r2_means.append(best_r2_curr)
    r2_stds.append(best_std_r2)

    results.append({
        "Num_Features": num_features,
        "Spearman_Mean": best_spearman_curr,
        "Spearman_Std": best_std_spearman,
        "RMSE_Mean": best_rmse_curr,
        "RMSE_Std": best_std_rmse,
        "MAE_Mean": best_mae_curr,
        "MAE_Std": best_std_mae,
        "R2_Mean": best_r2_curr,
        "R2_Std": best_std_r2,
        "Selected_Features": ",".join(best_features_curr)
    })

pbar.close()

# 保存结果
pd.DataFrame(results).to_csv("RR_Feature_Selection_Results.csv", index=False)
pd.DataFrame(all_param_results).to_csv("RR_All_Param_Results.csv", index=False)

# 最优模型评估（基于最佳一折）
X_selected_final = best_selector.transform(X_scaled)
X_train = X_selected_final[best_train_idx]
X_test = X_selected_final[best_test_idx]
y_train = y[best_train_idx]
y_test = y[best_test_idx]

final_model = Ridge(alpha=best_params["alpha"], tol=best_params["tol"])
final_model.fit(X_train, y_train)
y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

spearman_train, _ = spearmanr(y_train, y_train_pred)
spearman_test, _ = spearmanr(y_test, y_test_pred)
rmse_train = mean_squared_error(y_train, y_train_pred, squared=False)
rmse_test = mean_squared_error(y_test, y_test_pred, squared=False)
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# 展示结果
print("\n📊 最佳模型评估指标（基于 Spearman 最高一折）：")
print(f"▶ Train:  Spearman = {spearman_train:.3f}, RMSE = {rmse_train:.3f}, MAE = {mae_train:.3f}, R² = {r2_train:.3f}")
print(f"▶ Test:   Spearman = {spearman_test:.3f}, RMSE = {rmse_test:.3f}, MAE = {mae_test:.3f}, R² = {r2_test:.3f}")
print(f"▶ 最佳参数：alpha = {best_params['alpha']}, tol = {best_params['tol']}")
print(f"▶ 最佳特征数：{len(best_selected_features)}")
print("▶ 最佳特征：", ", ".join(best_selected_features))

# 保存最优模型参数
best_model_params = {
    "Best Alpha": best_params["alpha"],
    "Best Tol": best_params["tol"],
    "Best Num Features": len(best_selected_features),
    "Selected Features": ",".join(best_selected_features),
    "Spearman (Train)": spearman_train,
    "Spearman (Test)": spearman_test,
    "RMSE (Train)": rmse_train,
    "RMSE (Test)": rmse_test,
    "MAE (Train)": mae_train,
    "MAE (Test)": mae_test,
    "R2 (Train)": r2_train,
    "R2 (Test)": r2_test,
    "Train_idx": best_train_idx,
    "Test_idx": best_test_idx
}
pd.DataFrame([best_model_params]).to_csv("RR_Best_Model.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axs = plt.subplots(2, 2, figsize=(14, 12))
plt.subplots_adjust(wspace=0.3, hspace=0.35)  # 调整子图间距

# 图1：R² vs 特征数（左上）
axs[0, 0].errorbar(feature_counts, r2_means, fmt='-o', color='mediumseagreen')
axs[0, 0].set_title("R² vs Feature Count", fontsize=18)
axs[0, 0].set_xlabel("Number of Features", fontsize=16)
axs[0, 0].set_ylabel("R² Score", fontsize=16)
axs[0, 0].grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
axs[0, 0].tick_params(axis="both", direction="in", labelsize=14)
for spine in axs[0, 0].spines.values():
    spine.set_linewidth(2)

# 图2：Spearman vs 特征数（右上）
axs[0, 1].errorbar(feature_counts, spearman_means, fmt='-o', color='royalblue')
axs[0, 1].set_title("Spearman vs Feature Count", fontsize=18)
axs[0, 1].set_xlabel("Number of Features", fontsize=16)
axs[0, 1].set_ylabel("Spearman Coefficient", fontsize=16)
axs[0, 1].grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
axs[0, 1].tick_params(axis="both", direction="in", labelsize=14)
axs[0, 1].set_ylim(0.68,0.86)
# axs[0, 1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
for spine in axs[0, 1].spines.values():
    spine.set_linewidth(2)

# 图3：RMSE vs 特征数（左下）
axs[1, 0].errorbar(feature_counts, rmse_means, fmt='-o', color='tomato')
axs[1, 0].set_title("RMSE vs Feature Count", fontsize=18)
axs[1, 0].set_xlabel("Number of Features", fontsize=16)
axs[1, 0].set_ylabel("RMSE", fontsize=16)
axs[1, 0].grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
axs[1, 0].tick_params(axis="both", direction="in", labelsize=14)
for spine in axs[1, 0].spines.values():
    spine.set_linewidth(2)

# 图4：MAE vs 特征数（右下）
axs[1, 1].errorbar(feature_counts, mae_means, fmt='-o', color='crimson')
axs[1, 1].set_title("MAE vs Feature Count", fontsize=18)
axs[1, 1].set_xlabel("Number of Features", fontsize=16)
axs[1, 1].set_ylabel("MAE", fontsize=16)
axs[1, 1].grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
axs[1, 1].tick_params(axis="both", direction="in", labelsize=14)
axs[1, 1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
for spine in axs[1, 1].spines.values():
    spine.set_linewidth(2)

# 显示图像
plt.show()

In [ ]:
import matplotlib.ticker as ticker
import numpy as np  # 确保引入 np

# 图像配色
colors = {
    "R²": "mediumseagreen",
    "Spearman": "royalblue",
    "RMSE": "tomato",
    "MAE": "crimson"
}

# 格式化器字典（用于统一精度）
formatters = {
    "R²": ticker.FormatStrFormatter('%.2f'),
    "Spearman": ticker.FormatStrFormatter('%.2f'),
    "RMSE": ticker.FormatStrFormatter('%.2f'),
    "MAE": ticker.FormatStrFormatter('%.2f')
}

# 指标数据映射
metrics_data = {
    "R²": r2_means,
    "Spearman": spearman_means,
    "RMSE": rmse_means,
    "MAE": mae_means
}

# 指标组合（两两）
combinations = [
    ("R²", "Spearman"),
    ("R²", "RMSE"),
    ("R²", "MAE"),
    ("Spearman", "RMSE"),
    ("Spearman", "MAE"),
    ("RMSE", "MAE")
]

# 开始绘图
for i, (metric1, metric2) in enumerate(combinations):
    fig, ax1 = plt.subplots(figsize=(7.5, 6))

    # 第一个指标
    ax1.plot(feature_counts, metrics_data[metric1], marker="o", linestyle="-",
             color=colors[metric1], label=metric1)
    ax1.set_xlabel("Number of Selected Features", fontsize=18)
    ax1.set_ylabel(metric1, fontsize=18, color=colors[metric1])
    ax1.tick_params(axis='both', direction='in', labelsize=16)
    ax1.yaxis.label.set_color(colors[metric1])
    ax1.yaxis.set_major_formatter(formatters[metric1])
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)

    # 如果是 Spearman，则设置 y-ticks
    if metric1 == "RMSE":
        ax1.set_yticks(np.arange(0.7, 1.10 + 0.001, 0.1))

    # 第二个指标（右轴）
    ax2 = ax1.twinx()
    ax2.plot(feature_counts, metrics_data[metric2], marker="s", linestyle="--",
             color=colors[metric2], label=metric2)
    ax2.set_ylabel(metric2, fontsize=18, color=colors[metric2])
    ax2.tick_params(axis='y', direction='in', labelsize=16)
    ax2.yaxis.label.set_color(colors[metric2])
    ax2.yaxis.set_major_formatter(formatters[metric2])

    # 如果是 Spearman，则设置 y-ticks
    if metric2 == "RMSE":
        ax2.set_yticks(np.arange(0.75, 1.10 + 0.001, 0.05))

    # 美化边框线宽
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

    plt.title(f"{metric1} vs {metric2}", fontsize=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# 打印评价指标
print(f"Train Spearman = {spearman_train:.3f}, RMSE = {rmse_train:.3f}")
print(f"Test  Spearman = {spearman_test:.3f}, RMSE = {rmse_test:.3f}")

# 绘图
plt.figure(figsize=(7, 6.5))
plt.scatter(y_train, y_train_pred, color="c", label="Train", alpha=0.6)
plt.scatter(y_test, y_test_pred, color="indianred", label="Test", alpha=0.6)
plt.plot([min(y), max(y)], [min(y), max(y)], "k--", lw=2)

plt.xlabel("Ground Truth DM [Debye]", fontsize=18)
plt.ylabel("Predicted DM [Debye]", fontsize=18)
plt.title("Ground Truth vs. Predicted DM (Ridge Regression)", fontsize=20)
plt.legend(fontsize=16, loc="upper left")
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.tick_params(axis="both", direction="in", labelsize=16)

# 美化边框
ax = plt.gca()
for side in ["top", "right", "left", "bottom"]:
    ax.spines[side].set_linewidth(2)

plt.show()

In [ ]:
# 获取最终模型的特征重要性
feature_importance = final_model.coef_

# 对特征重要性进行排序
importance_sorted_idx = np.argsort(np.abs(feature_importance))[::-1]  # 按照特征重要性从大到小排序

# 获取排序后的特征名称和特征重要性
sorted_feature_names = [best_selected_features[i] for i in importance_sorted_idx]
sorted_importance = np.abs(feature_importance[importance_sorted_idx])

# 将排序后的特征名称和特征重要性保存到 CSV 文件
feature_importance_df_sorted = pd.DataFrame({
    "Feature": sorted_feature_names,
    "Importance": sorted_importance})

# 保存到 CSV 文件
feature_importance_df_sorted.to_csv("RR_Sorted_Feature_Importance.csv", index=False)

# 设置合适的图形尺寸
plt.figure(figsize=(10, 8))

# **调整排序，使最重要的特征在 Y 轴最底部**
sns.barplot(y=sorted_feature_names[::-1], x=sorted_importance[::-1], color='skyblue')

# 确保所有边框存在并加粗
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)  # 设置边框宽度为 2

# 设置 ticks 朝内
plt.tick_params(axis='x', direction='in', length=4, width=0.75)
plt.tick_params(axis='y', direction='in', length=4, width=0.75)

# 设置标签和标题
plt.xlabel('Feature Importance', fontsize=18)
plt.ylabel('Features', fontsize=18)
plt.title('Feature Importance of Ridge Regression Model', fontsize=20)

# 调整 ticks 字体大小
plt.xticks(fontsize=12)
plt.yticks(fontsize=8.5)

# **调整 Y 轴，使顶部和底部条形都留出空隙**
plt.ylim(-0.8, len(sorted_feature_names) - 0.2)  # 让第一条和最后一条都远离边框

# 调整图形布局以避免重叠
plt.tight_layout()
plt.grid(True, linestyle='-', linewidth=0.5, alpha=0.75)

# 显示图形
plt.show()

In [ ]:
import pickle
import shap

# 保存模型
with open("RR_Best_Model.pkl", "wb") as f:
    pickle.dump(final_model, f)

# 加载模型
with open("RR_Best_Model.pkl", "rb") as f:
    loaded_model = pickle.load(f)
    
# 特征选择与特征名
X_selected_all = best_selector.transform(X_scaled)
selected_feature_names = best_selected_features
X_selected_df = pd.DataFrame(X_selected_all, columns=selected_feature_names)

# 创建 LinearExplainer（专门用于线性模型，如 Ridge）
explainer = shap.LinearExplainer(loaded_model, X_selected_all, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_selected_all)

# summary_plot: bar 类型
shap.summary_plot(shap_values, features=X_selected_all, feature_names=selected_feature_names, plot_type="bar", show=False)
plt.gcf().set_size_inches(10, 8)
ax = plt.gca()
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
ax.set_title("Feature Importance", fontsize=16)
plt.tight_layout()
plt.show()

# summary_plot: dot 类型
shap.summary_plot(shap_values, features=X_selected_all, feature_names=selected_feature_names, show=False)
plt.gcf().set_size_inches(10, 8)
ax = plt.gca()
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
cbar = plt.gcf().axes[-1]
cbar.tick_params(labelsize=14)
ax.set_title("Feature Importance", fontsize=16)
plt.tight_layout()
plt.show()

# dependence_plot：每个特征与 SHAP 值的关系图
for feature in selected_feature_names:
    plt.figure()
    shap.dependence_plot(feature, shap_values, X_selected_df, interaction_index=None, show=False)
    plt.title(f"SHAP Dependence: {feature}", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# 读取数据并筛选
df = pd.read_csv("RR_All_Param_Results.csv")
df_filtered = df[df["Num_Features"] == 40].copy()

# 创建分类坐标映射
tol_values = sorted(df_filtered["Tol"].unique())
alpha_values = sorted(df_filtered["Alpha"].unique())

# 映射为 index（均匀间距）
df_filtered["tol_idx"] = df_filtered["Tol"].apply(lambda x: tol_values.index(x))
df_filtered["alpha_idx"] = df_filtered["Alpha"].apply(lambda x: alpha_values.index(x))

# 设置画布
plt.figure(figsize=(8, 6.5))

# 绘制散点图
sc = plt.scatter(
    df_filtered["tol_idx"],
    df_filtered["alpha_idx"],
    c=df_filtered["RMSE_Mean"],
    cmap="coolwarm",
    s=200,
    edgecolors='black',
    linewidths=0.5
)

# 添加 colorbar
cbar = plt.colorbar(sc)
cbar.set_label("RMSE", fontsize=16)
cbar.ax.tick_params(labelsize=14)
cbar.set_ticks([0.76, 0.77, 0.78, 0.79, 0.80])  

# 设置分类型坐标轴
plt.xticks(ticks=np.arange(len(tol_values)), labels=tol_values, fontsize=16)
plt.yticks(ticks=np.arange(len(alpha_values)), labels=alpha_values, fontsize=16)
plt.tick_params(axis="both", direction="in", labelsize=16)

# 设置标签与标题
plt.xlabel("Convergence tolerance", fontsize=18)
plt.ylabel("Alpha", fontsize=18)
plt.title("RR Hyperparameter Tuning - RMSE Distribution", fontsize=20)

# 加网格线
plt.grid(True, linestyle="--", alpha=0.7)

ax = plt.gca()
for side in ["top", "right", "left", "bottom"]:
    ax.spines[side].set_linewidth(2)

plt.tight_layout()
plt.show()